In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
DATA_PATH = "data"

ratings_path = DATA_PATH + "/rating.csv"
movies_path = DATA_PATH + "/movie.csv"

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

print(ratings.shape)
print(movies.shape)

(20000263, 4)
(27278, 3)


In [3]:
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"])
ratings["label"] = (ratings["rating"] >= 4).astype(int)

def extract_year(title):
    match = re.search(r"\((\d{4})\)", str(title))
    if match:
        return int(match.group(1))
    return np.nan

movies["movie_year"] = movies["title"].apply(extract_year)

print(ratings.head())
print(movies[["movieId", "title", "movie_year"]].head())

   userId  movieId  rating           timestamp  label
0       1        2     3.5 2005-04-02 23:53:47      0
1       1       29     3.5 2005-04-02 23:31:16      0
2       1       32     3.5 2005-04-02 23:33:39      0
3       1       47     3.5 2005-04-02 23:32:07      0
4       1       50     3.5 2005-04-02 23:29:40      0
   movieId                               title  movie_year
0        1                    Toy Story (1995)      1995.0
1        2                      Jumanji (1995)      1995.0
2        3             Grumpier Old Men (1995)      1995.0
3        4            Waiting to Exhale (1995)      1995.0
4        5  Father of the Bride Part II (1995)      1995.0


In [4]:
SAMPLE_SIZE = 200000
SAMPLE_SEED = 42

ratings_sampled = (
    ratings
    .groupby("label", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(
                len(x),
                int(SAMPLE_SIZE * len(x) / len(ratings))
            ),
            random_state=SAMPLE_SEED
        )
    )
    .sample(frac=1, random_state=SAMPLE_SEED)
    .reset_index(drop=True)
)

print(ratings_sampled.shape)

print(
    ratings_sampled["label"]
    .value_counts(normalize=True)
)

(199999, 5)
label
0    0.500238
1    0.499762
Name: proportion, dtype: float64


/tmp/ipykernel_1180/1729049423.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [5]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)

    train_indices = []
    test_indices = []

    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(frac=1, random_state=random_seed).reset_index(drop=True)
    test_df = df.loc[test_indices].sample(frac=1, random_state=random_seed).reset_index(drop=True)

    return train_df, test_df

In [6]:
def compute_train_statistics(train_df):
    global_mean = train_df["rating"].mean()
    global_like_ratio = train_df["label"].mean()

    user_stats = train_df.groupby("userId").agg(
        user_avg_rating=("rating", "mean"),
        user_rating_count=("rating", "count"),
        user_rating_std=("rating", "std"),
        user_like_count=("label", "sum"),
        user_first_rating_time=("timestamp", "min"),
        user_last_rating_time=("timestamp", "max")
    ).reset_index()

    user_stats["user_like_ratio"] = user_stats["user_like_count"] / user_stats["user_rating_count"]

    user_stats["user_rating_timespan"] = (
        user_stats["user_last_rating_time"] - user_stats["user_first_rating_time"]
    ).dt.days

    user_stats["user_avg_gap_days"] = (
        user_stats["user_rating_timespan"] / (user_stats["user_rating_count"] - 1)
    )

    user_stats["user_avg_gap_days"] = user_stats["user_avg_gap_days"].replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)

    user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0)

    user_stats = user_stats.drop(
        columns=["user_first_rating_time", "user_last_rating_time"]
    )

    item_stats = train_df.groupby("movieId").agg(
        item_avg_rating=("rating", "mean"),
        item_rating_count=("rating", "count"),
        item_rating_std=("rating", "std"),
        item_like_count=("label", "sum")
    ).reset_index()

    item_stats["item_like_ratio"] = item_stats["item_like_count"] / item_stats["item_rating_count"]
    item_stats["item_rating_std"] = item_stats["item_rating_std"].fillna(0)

    return user_stats, item_stats, global_mean, global_like_ratio

In [7]:
def build_feature_A(base_df, user_stats, item_stats, movies, global_mean, global_like_ratio):
    df = base_df.copy()
    df = df.merge(user_stats, on="userId", how="left")
    df = df.merge(item_stats, on="movieId", how="left")
    df = df.merge(movies[["movieId", "movie_year"]], on="movieId", how="left")
    df["global_mean"] = global_mean
    df["rating_year"] = df["timestamp"].dt.year
    df["movie_age_at_rating"] = df["rating_year"] - df["movie_year"]
    df["movie_age_at_rating"] = df["movie_age_at_rating"].fillna(0)

    df["user_avg_rating"] = df["user_avg_rating"].fillna(global_mean)
    df["user_rating_count"] = df["user_rating_count"].fillna(0)
    df["user_rating_std"] = df["user_rating_std"].fillna(0)
    df["user_like_count"] = df["user_like_count"].fillna(0)
    df["user_like_ratio"] = df["user_like_ratio"].fillna(global_like_ratio)
    df["user_rating_timespan"] = df["user_rating_timespan"].fillna(0)
    df["user_avg_gap_days"] = df["user_avg_gap_days"].fillna(0)
    df["item_avg_rating"] = df["item_avg_rating"].fillna(global_mean)
    df["item_rating_count"] = df["item_rating_count"].fillna(0)
    df["item_rating_std"] = df["item_rating_std"].fillna(0)
    df["item_like_count"] = df["item_like_count"].fillna(0)
    df["item_like_ratio"] = df["item_like_ratio"].fillna(global_like_ratio)
    feature_cols = [
        "user_avg_rating",
        "user_rating_count",
        "user_rating_std",
        "user_like_count",
        "user_like_ratio",
        "user_rating_timespan",
        "user_avg_gap_days",

        "item_avg_rating",
        "item_rating_count",
        "item_rating_std",
        "item_like_count",
        "item_like_ratio",

        "global_mean",
        "movie_age_at_rating"
    ]
    feature_A = df[
        ["userId", "movieId"] + feature_cols + ["label"]
    ].copy()
    return feature_A

In [8]:
OUTPUT_PATH = "FeatureA_Repeated"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10
TEST_RATIO = 0.2
BASE_SEED = 42

summary_records = []

for repeat_id in range(N_REPEATS):
    repeat_no = repeat_id + 1

    repeat_folder = os.path.join(
        OUTPUT_PATH,
        f"repeat_{repeat_no:02d}"
    )

    os.makedirs(repeat_folder, exist_ok=True)

    train_df, test_df = stratified_split_from_scratch(
        ratings_sampled,
        label_col="label",
        test_ratio=TEST_RATIO,
        random_seed=BASE_SEED + repeat_id
    )

    user_stats, item_stats, global_mean, global_like_ratio = compute_train_statistics(train_df)

    feature_A_train = build_feature_A(
        train_df,
        user_stats,
        item_stats,
        movies,
        global_mean,
        global_like_ratio
    )

    feature_A_test = build_feature_A(
        test_df,
        user_stats,
        item_stats,
        movies,
        global_mean,
        global_like_ratio
    )

    raw_train_path = os.path.join(repeat_folder, "raw_train.csv")
    raw_test_path = os.path.join(repeat_folder, "raw_test.csv")
    feature_train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    feature_test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df.to_csv(raw_train_path, index=False, encoding="utf-8-sig")
    test_df.to_csv(raw_test_path, index=False, encoding="utf-8-sig")
    feature_A_train.to_csv(feature_train_path, index=False, encoding="utf-8-sig")
    feature_A_test.to_csv(feature_test_path, index=False, encoding="utf-8-sig")

    summary_records.append({
        "repeat": repeat_no,
        "train_rows": len(feature_A_train),
        "test_rows": len(feature_A_test),
        "train_like_ratio": feature_A_train["label"].mean(),
        "test_like_ratio": feature_A_test["label"].mean(),
        "global_mean_from_train": global_mean,
        "feature_count_excluding_ids_and_label": feature_A_train.shape[1] - 3,
        "feature_train_path": feature_train_path,
        "feature_test_path": feature_test_path
    })

    print(f"Repeat {repeat_no:02d} saved")
    print("Train Feature A:", feature_A_train.shape)
    print("Test Feature A:", feature_A_test.shape)
    print("Global mean from train:", round(global_mean, 4))
    print("-" * 40)

Repeat 01 saved
Train Feature A: (160000, 17)
Test Feature A: (39999, 17)
Global mean from train: 3.5275
----------------------------------------
Repeat 02 saved
Train Feature A: (160000, 17)
Test Feature A: (39999, 17)
Global mean from train: 3.5268
----------------------------------------
Repeat 03 saved
Train Feature A: (160000, 17)
Test Feature A: (39999, 17)
Global mean from train: 3.5263
----------------------------------------
Repeat 04 saved
Train Feature A: (160000, 17)
Test Feature A: (39999, 17)
Global mean from train: 3.5268
----------------------------------------
Repeat 05 saved
Train Feature A: (160000, 17)
Test Feature A: (39999, 17)
Global mean from train: 3.5269
----------------------------------------
Repeat 06 saved
Train Feature A: (160000, 17)
Test Feature A: (39999, 17)
Global mean from train: 3.5257
----------------------------------------
Repeat 07 saved
Train Feature A: (160000, 17)
Test Feature A: (39999, 17)
Global mean from train: 3.527
--------------------

In [9]:
summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(OUTPUT_PATH, "feature_A_split_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved summary to:")
print(summary_path)

summary_df

Saved summary to:
FeatureA_Repeated/feature_A_split_summary.csv


,repeat,train_rows,test_rows,train_like_ratio,test_like_ratio,global_mean_from_train,feature_count_excluding_ids_and_label,feature_train_path,feature_test_path
0,1,160000,39999,0.499762,0.499762,3.527459,14,FeatureA_Repeated/repeat_01/feature_A_train.csv,FeatureA_Repeated/repeat_01/feature_A_test.csv
1,2,160000,39999,0.499762,0.499762,3.526838,14,FeatureA_Repeated/repeat_02/feature_A_train.csv,FeatureA_Repeated/repeat_02/feature_A_test.csv
2,3,160000,39999,0.499762,0.499762,3.526278,14,FeatureA_Repeated/repeat_03/feature_A_train.csv,FeatureA_Repeated/repeat_03/feature_A_test.csv
3,4,160000,39999,0.499762,0.499762,3.526825,14,FeatureA_Repeated/repeat_04/feature_A_train.csv,FeatureA_Repeated/repeat_04/feature_A_test.csv
4,5,160000,39999,0.499762,0.499762,3.526869,14,FeatureA_Repeated/repeat_05/feature_A_train.csv,FeatureA_Repeated/repeat_05/feature_A_test.csv
5,6,160000,39999,0.499762,0.499762,3.525725,14,FeatureA_Repeated/repeat_06/feature_A_train.csv,FeatureA_Repeated/repeat_06/feature_A_test.csv
6,7,160000,39999,0.499762,0.499762,3.527038,14,FeatureA_Repeated/repeat_07/feature_A_train.csv,FeatureA_Repeated/repeat_07/feature_A_test.csv
7,8,160000,39999,0.499762,0.499762,3.525859,14,FeatureA_Repeated/repeat_08/feature_A_train.csv,FeatureA_Repeated/repeat_08/feature_A_test.csv
8,9,160000,39999,0.499762,0.499762,3.526772,14,FeatureA_Repeated/repeat_09/feature_A_train.csv,FeatureA_Repeated/repeat_09/feature_A_test.csv
9,10,160000,39999,0.499762,0.499762,3.525478,14,FeatureA_Repeated/repeat_10/feature_A_train.csv,FeatureA_Repeated/repeat_10/feature_A_test.csv


In [10]:
preview_train_path = os.path.join(OUTPUT_PATH, "repeat_01", "feature_A_train.csv")
preview_test_path = os.path.join(OUTPUT_PATH, "repeat_01", "feature_A_test.csv")

preview_train = pd.read_csv(preview_train_path)
preview_test = pd.read_csv(preview_test_path)

print("Repeat 01 Feature A train shape:", preview_train.shape)
print("Repeat 01 Feature A test shape:", preview_test.shape)

print("Number of Feature A columns excluding IDs and label:", preview_train.shape[1] - 3)

preview_train.head()

Repeat 01 Feature A train shape: (160000, 17)
Repeat 01 Feature A test shape: (39999, 17)
Number of Feature A columns excluding IDs and label: 14


,userId,movieId,user_avg_rating,user_rating_count,user_rating_std,user_like_count,user_like_ratio,user_rating_timespan,user_avg_gap_days,item_avg_rating,item_rating_count,item_rating_std,item_like_count,item_like_ratio,global_mean,movie_age_at_rating,label
0,5176,2403,3.875000,4,0.629153,3,0.750000,0,0.0,3.565789,38,0.815153,17,0.447368,3.527459,22.0,0
1,134299,3693,4.571429,7,0.731925,6,0.857143,12,2.0,2.928571,7,1.484042,2,0.285714,3.527459,21.0,1
2,44717,2580,3.250000,4,0.500000,1,0.250000,3,1.0,3.701493,67,0.961610,44,0.656716,3.527459,1.0,1
3,87945,198,4.500000,2,0.707107,2,1.000000,0,0.0,3.417808,73,0.931746,34,0.465753,3.527459,5.0,1
4,54633,5025,3.500000,5,1.118034,2,0.400000,0,0.0,3.159091,22,0.836466,5,0.227273,3.527459,4.0,0
